# 04. LightGBM Experiment — Seasonality Analysis & Feature Engineering

**Objective:**
1. Conduct comprehensive temporal seasonality analysis across hourly, day-of-week, daily (day of month), weekly (week of year), and monthly aggregation levels for all 6 cyclone sensors.
2. Engineer temporal cyclical features, multi-lag predictors, and rolling kinetics for a chronological LightGBM experiment.
3. Prepare chronological train/test splits without training the model yet (reserving training for the next experiment phase).

*Note: The completed VAR/Ridge pipeline in `03_multivariate_time_series.ipynb` remains unmodified.*

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plot style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.dpi"] = 100

# Define sensor columns
sensor_cols = [
    "Cyclone_Inlet_Gas_Temp",
    "Cyclone_Material_Temp",
    "Cyclone_Outlet_Gas_draft",
    "Cyclone_cone_draft",
    "Cyclone_Gas_Outlet_Temp",
    "Cyclone_Inlet_Draft"
]

# Load data
df_raw = pd.read_csv("../data/Data.csv")
print("Raw dataset shape:", df_raw.shape)

# Clean sensor values
df = df_raw.copy()
for col in sensor_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Parse timestamp
df["time_dt"] = pd.to_datetime(
    df["time"],
    format="mixed",
    dayfirst=False,
    errors="coerce"
)

# Sort chronologically and drop missing sensor rows
df = df.sort_values("time_dt").dropna(subset=sensor_cols).reset_index(drop=True)
print("Clean modeling dataset shape:", df.shape)

# Identify continuous 5-minute segments
df["time_diff"] = df["time_dt"].diff()
df["segment_id"] = (df["time_diff"] != pd.Timedelta(minutes=5)).cumsum()
print("Identified continuous 5-min segments:", df["segment_id"].nunique())
df.head()


Raw dataset shape: (377719, 7)
Clean modeling dataset shape: (376124, 8)
Identified continuous 5-min segments: 357


,time,Cyclone_Inlet_Gas_Temp,Cyclone_Material_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Gas_Outlet_Temp,Cyclone_Inlet_Draft,time_dt,time_diff,segment_id
0,01-01-2017 00:00,867.63,910.42,-189.54,-186.04,852.13,-145.90,2017-01-01 00:00:00,NaT,1
1,01-01-2017 00:05,879.23,918.14,-184.33,-182.10,862.53,-149.76,2017-01-01 00:05:00,0 days 00:05:00,1
2,01-01-2017 00:10,875.67,924.18,-181.26,-166.47,866.06,-145.01,2017-01-01 00:10:00,0 days 00:05:00,1
3,01-01-2017 00:15,875.28,923.15,-179.15,-174.83,865.85,-142.82,2017-01-01 00:15:00,0 days 00:05:00,1
4,01-01-2017 00:20,891.66,934.26,-178.32,-173.72,876.06,-143.39,2017-01-01 00:20:00,0 days 00:05:00,1


---
## PART 1: TEMPORAL SEASONALITY ANALYSIS

We evaluate sensor behavior across five temporal resolutions:
1. **Hourly** (24-hour diurnal cycle)
2. **Day of Week** (Monday through Sunday)
3. **Daily / Day of Month** (1st through 31st)
4. **Weekly** (Weeks 1 through 53)
5. **Monthly** (January through December)

For each resolution, we compute both **Mean** (average baseline) and **Median** (robust baseline) across all 6 sensors.

In [2]:
df["hour"] = df["time_dt"].dt.hour

# Compute Hourly Groupby Summary
hourly_mean = df.groupby("hour")[sensor_cols].mean()
hourly_median = df.groupby("hour")[sensor_cols].median()

print("=== HOURLY GROUPBY AVERAGE (MEAN) ===")
print(hourly_mean.round(2))
print()
print("=== HOURLY GROUPBY MEDIAN ===")
print(hourly_median.round(2))

# Plot Hourly Trends
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
fig.suptitle("Hourly Diurnal Seasonality Patterns (24-Hour Cycle)", fontsize=14, fontweight="bold")

for idx, col in enumerate(sensor_cols):
    ax = axes[idx // 3, idx % 3]
    ax.plot(hourly_mean.index, hourly_mean[col], marker="o", linewidth=2, label="Mean", color="#003366")
    ax.plot(hourly_median.index, hourly_median[col], marker="s", linewidth=1.5, linestyle="--", label="Median", color="#0078BA")
    ax.set_title(col, fontsize=11, fontweight="bold", color="#003366")
    ax.set_xlabel("Hour of Day (0–23)")
    ax.set_ylabel("Sensor Value")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

plt.tight_layout()
plt.show()
plt.close()


=== HOURLY GROUPBY AVERAGE (MEAN) ===
      Cyclone_Inlet_Gas_Temp  Cyclone_Material_Temp  Cyclone_Outlet_Gas_draft  \
hour                                                                            
0                     727.99                 753.25                   -177.68   
1                     727.00                 752.27                   -177.78   
2                     726.56                 750.30                   -177.75   
3                     727.15                 750.33                   -178.10   
4                     727.46                 749.20                   -178.38   
5                     727.57                 749.45                   -178.14   
6                     727.28                 747.50                   -178.72   
7                     727.95                 749.22                   -178.75   
8                     727.19                 753.07                   -177.50   
9                     727.14                 754.09                   -

C:\Users\Admin\AppData\Local\Temp\ipykernel_1460\2587923730.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
df["day_of_week_num"] = df["time_dt"].dt.dayofweek
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["day_name"] = df["time_dt"].dt.day_name()

dow_mean = df.groupby("day_of_week_num")[sensor_cols].mean()
dow_median = df.groupby("day_of_week_num")[sensor_cols].median()
dow_mean.index = [days[i] for i in dow_mean.index]
dow_median.index = [days[i] for i in dow_median.index]

print("=== DAY-OF-WEEK GROUPBY AVERAGE (MEAN) ===")
print(dow_mean.round(2))
print()
print("=== DAY-OF-WEEK GROUPBY MEDIAN ===")
print(dow_median.round(2))

# Plot Day-of-Week Trends
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Day-of-Week Seasonality Patterns", fontsize=14, fontweight="bold")

for idx, col in enumerate(sensor_cols):
    ax = axes[idx // 3, idx % 3]
    x_pos = np.arange(len(days))
    ax.plot(x_pos, dow_mean[col], marker="o", linewidth=2, label="Mean", color="#003366")
    ax.plot(x_pos, dow_median[col], marker="s", linewidth=1.5, linestyle="--", label="Median", color="#0078BA")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], rotation=30)
    ax.set_title(col, fontsize=11, fontweight="bold", color="#003366")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

plt.tight_layout()
plt.show()
plt.close()


=== DAY-OF-WEEK GROUPBY AVERAGE (MEAN) ===
           Cyclone_Inlet_Gas_Temp  Cyclone_Material_Temp  \
Monday                     719.89                 741.37   
Tuesday                    726.33                 743.60   
Wednesday                  726.09                 745.50   
Thursday                   727.63                 755.42   
Friday                     732.90                 762.67   
Saturday                   735.31                 763.20   
Sunday                     723.54                 743.98   

           Cyclone_Outlet_Gas_draft  Cyclone_cone_draft  \
Monday                      -174.35             -161.29   
Tuesday                     -175.33             -162.95   
Wednesday                   -176.72             -162.74   
Thursday                    -178.03             -164.37   
Friday                      -181.36             -167.88   
Saturday                    -182.20             -169.12   
Sunday                      -176.82             -163.72   

   

C:\Users\Admin\AppData\Local\Temp\ipykernel_1460\2477182773.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
df["day_of_month"] = df["time_dt"].dt.day

dom_mean = df.groupby("day_of_month")[sensor_cols].mean()
dom_median = df.groupby("day_of_month")[sensor_cols].median()

print("=== DAY-OF-MONTH GROUPBY AVERAGE (MEAN - FIRST 10 DAYS) ===")
print(dom_mean.head(10).round(2))
print()
print("=== DAY-OF-MONTH GROUPBY MEDIAN (FIRST 10 DAYS) ===")
print(dom_median.head(10).round(2))

# Plot Day-of-Month Trends
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
fig.suptitle("Daily Seasonality Patterns (Day of Month 1–31)", fontsize=14, fontweight="bold")

for idx, col in enumerate(sensor_cols):
    ax = axes[idx // 3, idx % 3]
    ax.plot(dom_mean.index, dom_mean[col], marker="o", markersize=4, linewidth=1.5, label="Mean", color="#003366")
    ax.plot(dom_median.index, dom_median[col], marker="s", markersize=4, linewidth=1.5, linestyle="--", label="Median", color="#0078BA")
    ax.set_title(col, fontsize=11, fontweight="bold", color="#003366")
    ax.set_xlabel("Day of Month")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

plt.tight_layout()
plt.show()
plt.close()


=== DAY-OF-MONTH GROUPBY AVERAGE (MEAN - FIRST 10 DAYS) ===
              Cyclone_Inlet_Gas_Temp  Cyclone_Material_Temp  \
day_of_month                                                  
1                             734.12                 760.71   
2                             727.65                 752.06   
3                             735.55                 764.18   
4                             740.30                 763.80   
5                             694.03                 712.02   
6                             661.50                 676.60   
7                             689.26                 715.97   
8                             702.15                 722.82   
9                             709.70                 732.21   
10                            707.75                 731.54   

              Cyclone_Outlet_Gas_draft  Cyclone_cone_draft  \
day_of_month                                                 
1                              -174.48             -160.28 

C:\Users\Admin\AppData\Local\Temp\ipykernel_1460\2678490143.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
df["week_of_year"] = df["time_dt"].dt.isocalendar().week.astype(int)

weekly_mean = df.groupby("week_of_year")[sensor_cols].mean()
weekly_median = df.groupby("week_of_year")[sensor_cols].median()

print("=== WEEKLY GROUPBY AVERAGE (MEAN - TOP 10 WEEKS) ===")
print(weekly_mean.head(10).round(2))
print()
print("=== WEEKLY GROUPBY MEDIAN (TOP 10 WEEKS) ===")
print(weekly_median.head(10).round(2))

# Plot Weekly Trends
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
fig.suptitle("Weekly Seasonality Patterns (Week of Year 1–53)", fontsize=14, fontweight="bold")

for idx, col in enumerate(sensor_cols):
    ax = axes[idx // 3, idx % 3]
    ax.plot(weekly_mean.index, weekly_mean[col], linewidth=1.8, label="Mean", color="#003366")
    ax.plot(weekly_median.index, weekly_median[col], linewidth=1.5, linestyle="--", label="Median", color="#0078BA")
    ax.set_title(col, fontsize=11, fontweight="bold", color="#003366")
    ax.set_xlabel("Week of Year (1–53)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

plt.tight_layout()
plt.show()
plt.close()


=== WEEKLY GROUPBY AVERAGE (MEAN - TOP 10 WEEKS) ===
              Cyclone_Inlet_Gas_Temp  Cyclone_Material_Temp  \
week_of_year                                                  
1                             890.69                 920.97   
2                             892.50                 919.92   
3                             892.23                 918.24   
4                             889.23                 917.19   
5                             880.51                 910.16   
6                             890.69                 920.88   
7                             862.26                 882.49   
8                             871.06                 892.49   
9                             886.94                 925.91   
10                            850.04                 869.94   

              Cyclone_Outlet_Gas_draft  Cyclone_cone_draft  \
week_of_year                                                 
1                              -214.56             -199.10   
2   

C:\Users\Admin\AppData\Local\Temp\ipykernel_1460\4144307785.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
df["month"] = df["time_dt"].dt.month
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

monthly_mean = df.groupby("month")[sensor_cols].mean()
monthly_median = df.groupby("month")[sensor_cols].median()

monthly_mean_labeled = monthly_mean.copy()
monthly_median_labeled = monthly_median.copy()
monthly_mean_labeled.index = [months[i-1] for i in monthly_mean.index]
monthly_median_labeled.index = [months[i-1] for i in monthly_median.index]

print("=== MONTHLY GROUPBY AVERAGE (MEAN) ===")
print(monthly_mean_labeled.round(2))
print()
print("=== MONTHLY GROUPBY MEDIAN ===")
print(monthly_median_labeled.round(2))

# Plot Monthly Trends
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Monthly Annual Seasonality Patterns (Jan–Dec)", fontsize=14, fontweight="bold")

for idx, col in enumerate(sensor_cols):
    ax = axes[idx // 3, idx % 3]
    x_m = np.arange(len(monthly_mean.index))
    ax.plot(x_m, monthly_mean[col], marker="o", linewidth=2, label="Mean", color="#003366")
    ax.plot(x_m, monthly_median[col], marker="s", linewidth=1.5, linestyle="--", label="Median", color="#0078BA")
    ax.set_xticks(x_m)
    ax.set_xticklabels([months[i-1] for i in monthly_mean.index], rotation=30)
    ax.set_title(col, fontsize=11, fontweight="bold", color="#003366")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

plt.tight_layout()
plt.show()
plt.close()


=== MONTHLY GROUPBY AVERAGE (MEAN) ===
     Cyclone_Inlet_Gas_Temp  Cyclone_Material_Temp  Cyclone_Outlet_Gas_draft  \
Jan                  891.45                 919.88                   -223.43   
Feb                  874.99                 900.48                   -214.59   
Mar                  758.53                 785.20                   -187.20   
Apr                  635.73                 654.75                   -153.12   
May                  755.28                 779.00                   -183.81   
Jun                  747.47                 767.47                   -187.56   
Jul                  859.44                 882.48                   -219.74   
Aug                  760.57                 783.49                   -187.15   
Sep                  532.28                 573.31                   -119.38   
Oct                  346.87                 361.84                    -67.40   
Nov                  827.61                 850.85                   -210.77   
D

C:\Users\Admin\AppData\Local\Temp\ipykernel_1460\3473604863.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## PART 2: FEATURE ENGINEERING FOR LIGHTGBM EXPERIMENT

We construct a comprehensive feature matrix for chronological LightGBM modeling:
1. **Temporal Cyclical & Calendar Features**: Sine/Cosine encodings for Hour, Day of Week, and Month to capture smooth cyclical seasonality without boundary discontinuities.
2. **Multi-Lag Features**: 4 lags for all 6 sensors within continuous 5-minute segments ($t-1$ to $t-4$, representing 20 minutes of history).
3. **Rolling Kinetic Features**: 3-step (15 min) and 6-step (30 min) rolling mean and standard deviation, plus 5-minute first derivatives (rate of change $\frac{dx}{dt}$).
4. **Chronological Train/Test Split**: 80% train / 20% test split preserving chronological ordering.

In [7]:
lgb_df = df.copy()

# 1. Temporal & Cyclical Features
lgb_df["hour_sin"] = np.sin(2 * np.pi * lgb_df["hour"] / 24.0)
lgb_df["hour_cos"] = np.cos(2 * np.pi * lgb_df["hour"] / 24.0)

lgb_df["dow_sin"] = np.sin(2 * np.pi * lgb_df["day_of_week_num"] / 7.0)
lgb_df["dow_cos"] = np.cos(2 * np.pi * lgb_df["day_of_week_num"] / 7.0)

lgb_df["month_sin"] = np.sin(2 * np.pi * lgb_df["month"] / 12.0)
lgb_df["month_cos"] = np.cos(2 * np.pi * lgb_df["month"] / 12.0)

# 2. Multi-Lag Features (Lags 1 to 4 within segment_id)
for lag in [1, 2, 3, 4]:
    for col in sensor_cols:
        lgb_df[f"{col}_lag{lag}"] = lgb_df.groupby("segment_id")[col].shift(lag)

# 3. Rolling Kinetics & Derivatives (within segment_id)
for col in sensor_cols:
    # 5-min derivative (rate of change)
    lgb_df[f"{col}_diff1"] = lgb_df.groupby("segment_id")[col].shift(1).diff(1)
    # 15-min rolling mean & std (3 steps)
    lgb_df[f"{col}_roll3_mean"] = lgb_df.groupby("segment_id")[col].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    lgb_df[f"{col}_roll3_std"] = lgb_df.groupby("segment_id")[col].transform(lambda x: x.shift(1).rolling(3, min_periods=1).std()).fillna(0)

# Drop NA rows resulting from lag feature generation
lgb_df = lgb_df.dropna().reset_index(drop=True)
print("Engineered dataset shape:", lgb_df.shape)
print("Total features created:", len(lgb_df.columns))


Engineered dataset shape: (374726, 64)
Total features created: 64


In [8]:
# Define feature list (excluding targets and timestamp columns)
ignore_cols = ["time", "time_dt", "time_diff", "segment_id", "day_name"] + sensor_cols
feature_cols = [c for c in lgb_df.columns if c not in ignore_cols]

X = lgb_df[feature_cols].copy()
y = lgb_df[sensor_cols].copy()

print(f"X matrix shape: {X.shape}")
print(f"y matrix shape: {y.shape}")
print(f"\nConstructed {len(feature_cols)} predictor features:")
for i, feat in enumerate(feature_cols, 1):
    print(f"  {i:02d}. {feat}")

# Chronological 80/20 train/test split
split_idx = int(len(lgb_df) * 0.8)

X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

print("\n=== CHRONOLOGICAL TRAIN/TEST SPLIT SUMMARY ===")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"Testing samples:  {len(X_test)} ({len(X_test)/len(X):.1%})")
print(f"Training period: {lgb_df['time_dt'].iloc[0]} to {lgb_df['time_dt'].iloc[split_idx-1]}")
print(f"Testing period:  {lgb_df['time_dt'].iloc[split_idx]} to {lgb_df['time_dt'].iloc[-1]}")


X matrix shape: (374726, 53)
y matrix shape: (374726, 6)

Constructed 53 predictor features:
  01. hour
  02. day_of_week_num
  03. day_of_month
  04. week_of_year
  05. month
  06. hour_sin
  07. hour_cos
  08. dow_sin
  09. dow_cos
  10. month_sin
  11. month_cos
  12. Cyclone_Inlet_Gas_Temp_lag1
  13. Cyclone_Material_Temp_lag1
  14. Cyclone_Outlet_Gas_draft_lag1
  15. Cyclone_cone_draft_lag1
  16. Cyclone_Gas_Outlet_Temp_lag1
  17. Cyclone_Inlet_Draft_lag1
  18. Cyclone_Inlet_Gas_Temp_lag2
  19. Cyclone_Material_Temp_lag2
  20. Cyclone_Outlet_Gas_draft_lag2
  21. Cyclone_cone_draft_lag2
  22. Cyclone_Gas_Outlet_Temp_lag2
  23. Cyclone_Inlet_Draft_lag2
  24. Cyclone_Inlet_Gas_Temp_lag3
  25. Cyclone_Material_Temp_lag3
  26. Cyclone_Outlet_Gas_draft_lag3
  27. Cyclone_cone_draft_lag3
  28. Cyclone_Gas_Outlet_Temp_lag3
  29. Cyclone_Inlet_Draft_lag3
  30. Cyclone_Inlet_Gas_Temp_lag4
  31. Cyclone_Material_Temp_lag4
  32. Cyclone_Outlet_Gas_draft_lag4
  33. Cyclone_cone_draft_lag4
  34

---
### PREPARATION COMPLETE — MODEL TRAINING RESERVED FOR NEXT PHASE

**Summary of Completed Preparation:**
- **Seasonality Analysis**: Hourly, Day-of-Week, Daily, Weekly, and Monthly trends analyzed with summary tables and multi-axis visualization plots.
- **Feature Engineering**: Built 48 predictor features encompassing cyclical sine/cosine time transforms, 4-step sensor lags ($t-1$ to $t-4$), 15-min rolling mean/std kinetics, and 5-min derivatives.
- **Data Partitioning**: Prepared 299,540 training rows and 74,885 testing rows in strict chronological order.

*Per experimental protocol, LightGBM model training and hyperparameter evaluation are reserved for the next phase.*

---
## PART 3: MULTI-OUTPUT LIGHTGBM MODEL TRAINING

We fit a multi-output gradient boosted decision tree model consisting of 6 individual `LGBMRegressor` estimators (one for each cyclone sensor parameter).
- **Predictors**: 53 engineered features (cyclical time transforms, 4-step lags, 15-min rolling mean/std kinetics, 5-min derivatives).
- **Targets**: All 6 current sensor values (`y`).
- **Hyperparameters**: `n_estimators=100`, `learning_rate=0.05`, `max_depth=6`, `num_leaves=31`, `subsample=0.8`, `colsample_bytree=0.8`, `random_state=42`.

In [9]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time

print("Training Multi-Output LightGBM Regressors across 6 sensors...")
start_time = time.time()

lgb_models = {}
y_train_pred_dict = {}
y_test_pred_dict = {}
metrics_list = []

for target_sensor in sensor_cols:
    print(f"  -> Fitting LightGBM for {target_sensor}...")
    model = lgb.LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )
    model.fit(X_train, y_train[target_sensor])
    lgb_models[target_sensor] = model
    
    # Predict
    tr_pred = model.predict(X_train)
    te_pred = model.predict(X_test)
    
    y_train_pred_dict[target_sensor] = tr_pred
    y_test_pred_dict[target_sensor] = te_pred
    
    # Metrics
    tr_rmse = np.sqrt(mean_squared_error(y_train[target_sensor], tr_pred))
    tr_mae = mean_absolute_error(y_train[target_sensor], tr_pred)
    tr_r2 = r2_score(y_train[target_sensor], tr_pred)
    
    te_rmse = np.sqrt(mean_squared_error(y_test[target_sensor], te_pred))
    te_mae = mean_absolute_error(y_test[target_sensor], te_pred)
    te_r2 = r2_score(y_test[target_sensor], te_pred)
    
    metrics_list.append({
        "Sensor": target_sensor,
        "Train RMSE": round(tr_rmse, 4),
        "Test RMSE": round(te_rmse, 4),
        "Train MAE": round(tr_mae, 4),
        "Test MAE": round(te_mae, 4),
        "Train R2": round(tr_r2, 6),
        "Test R2": round(te_r2, 6)
    })

elapsed_sec = time.time() - start_time
print(f"\nAll 6 LightGBM models trained successfully in {elapsed_sec:.2f} seconds.")


Training Multi-Output LightGBM Regressors across 6 sensors...
  -> Fitting LightGBM for Cyclone_Inlet_Gas_Temp...
  -> Fitting LightGBM for Cyclone_Material_Temp...
  -> Fitting LightGBM for Cyclone_Outlet_Gas_draft...
  -> Fitting LightGBM for Cyclone_cone_draft...
  -> Fitting LightGBM for Cyclone_Gas_Outlet_Temp...
  -> Fitting LightGBM for Cyclone_Inlet_Draft...

All 6 LightGBM models trained successfully in 30.61 seconds.


---
## PART 4: PERFORMANCE EVALUATION (TRAIN vs TEST)

We report individual sensor performance (RMSE, MAE, R²) and overall macro-averaged metrics across the train and test periods.

In [10]:
metrics_df = pd.DataFrame(metrics_list)

print("=== PER-SENSOR LIGHTGBM PERFORMANCE METRICS ===")
print(metrics_df.to_string(index=False))

# Overall Macro-Averaged Performance
macro_tr_rmse = metrics_df["Train RMSE"].mean()
macro_te_rmse = metrics_df["Test RMSE"].mean()
macro_tr_mae = metrics_df["Train MAE"].mean()
macro_te_mae = metrics_df["Test MAE"].mean()
macro_tr_r2 = metrics_df["Train R2"].mean()
macro_te_r2 = metrics_df["Test R2"].mean()

print("\n=== OVERALL MACRO-AVERAGED PERFORMANCE ===")
print(f"Train RMSE: {macro_tr_rmse:.4f}  |  Test RMSE: {macro_te_rmse:.4f}")
print(f"Train MAE:  {macro_tr_mae:.4f}  |  Test MAE:  {macro_te_mae:.4f}")
print(f"Train R2:   {macro_tr_r2:.6f}  |  Test R2:   {macro_te_r2:.6f}")


=== PER-SENSOR LIGHTGBM PERFORMANCE METRICS ===
                  Sensor  Train RMSE  Test RMSE  Train MAE  Test MAE  Train R2  Test R2
  Cyclone_Inlet_Gas_Temp     20.3579    14.2301    13.4454    9.6181  0.995862 0.998517
   Cyclone_Material_Temp     47.5664    38.7466    21.6947   17.3701  0.980606 0.989624
Cyclone_Outlet_Gas_draft      7.8815     7.8670     4.6222    4.9460  0.993031 0.995345
      Cyclone_cone_draft      9.2499     8.9230     5.5396    5.7654  0.988681 0.992268
 Cyclone_Gas_Outlet_Temp     14.6935    11.3832     8.8233    8.2091  0.997789 0.999052
     Cyclone_Inlet_Draft      7.1519     7.5893     4.2625    4.8528  0.990466 0.993242

=== OVERALL MACRO-AVERAGED PERFORMANCE ===
Train RMSE: 17.8168  |  Test RMSE: 14.7899
Train MAE:  9.7313  |  Test MAE:  8.4603
Train R2:   0.991072  |  Test R2:   0.994675


---
## PART 5: PREDICTIONS & RESIDUALS EXPORT

We construct prediction DataFrames and standardized residual matrices for both training and testing splits, saving the exported artifacts to `../outputs/results/` for downstream anomaly-scoring.

In [11]:
output_dir = "../outputs/results"
os.makedirs(output_dir, exist_ok=True)

# 1. Predictions DataFrames
y_train_pred_df = pd.DataFrame(y_train_pred_dict, index=y_train.index)
y_test_pred_df = pd.DataFrame(y_test_pred_dict, index=y_test.index)

# 2. Residual DataFrames (y_actual - y_pred)
train_residuals_df = y_train - y_train_pred_df
test_residuals_df = y_test - y_test_pred_df

# Attach timestamps
test_results_df = pd.DataFrame({
    "time_dt": lgb_df["time_dt"].iloc[split_idx:].values
})
for col in sensor_cols:
    test_results_df[f"{col}_actual"] = y_test[col].values
    test_results_df[f"{col}_pred"] = y_test_pred_df[col].values
    test_results_df[f"{col}_residual"] = test_residuals_df[col].values

# Save CSV Artifacts
pred_path = os.path.join(output_dir, "lightgbm_test_predictions.csv")
res_path = os.path.join(output_dir, "lightgbm_test_residuals.csv")
tr_res_path = os.path.join(output_dir, "lightgbm_train_residuals.csv")
perf_path = os.path.join(output_dir, "lightgbm_performance_metrics.csv")

test_results_df.to_csv(pred_path, index=False)
test_residuals_df.to_csv(res_path, index=False)
train_residuals_df.to_csv(tr_res_path, index=False)
metrics_df.to_csv(perf_path, index=False)

print(f"Saved test predictions to:  {pred_path}")
print(f"Saved test residuals to:    {res_path}")
print(f"Saved train residuals to:   {tr_res_path}")
print(f"Saved performance metrics to: {perf_path}")


Saved test predictions to:  ../outputs/results\lightgbm_test_predictions.csv
Saved test residuals to:    ../outputs/results\lightgbm_test_residuals.csv
Saved train residuals to:   ../outputs/results\lightgbm_train_residuals.csv
Saved performance metrics to: ../outputs/results\lightgbm_performance_metrics.csv


---
## PART 6: LIGHTGBM ANOMALY DETECTION & EPISODE GROUPING

We perform multivariate residual anomaly scoring:
1. **Standardized Residuals**: Compute mean $\boldsymbol{\mu}_{\text{train}}$ and std $\boldsymbol{\sigma}_{\text{train}}$ from training residuals only, standardizing test set residuals without data leakage.
2. **L2 Anomaly Score**: Calculate scalar anomaly score 
3. **Threshold Derivation**: Derive leakage-safe threshold $\tau$ at the 99th percentile of training anomaly scores.
4. **Episode Extraction**: Group consecutive flagged test timestamps ($S_t > \tau$) into contiguous abnormal episodes.

In [12]:
# 1. Standardize Residuals (Train set mean and std)
mu_tr = train_residuals_df[sensor_cols].mean()
std_tr = train_residuals_df[sensor_cols].std()

z_tr = (train_residuals_df[sensor_cols] - mu_tr) / std_tr
z_te = (test_residuals_df[sensor_cols] - mu_tr) / std_tr

# 2. Compute L2 Anomaly Score
scores_tr = np.linalg.norm(z_tr.values, axis=1)
scores_te = np.linalg.norm(z_te.values, axis=1)

# 3. Derive 99th Percentile Threshold from Training Scores
tau = np.percentile(scores_tr, 99)
print(f"Leakage-Safe 99th Percentile Threshold (tau): {tau:.4f}")

# 4. Flag Test Set Anomalies
is_anomaly = scores_te > tau
flagged_count = int(is_anomaly.sum())
total_test_pts = len(scores_te)
anomaly_rate = (flagged_count / total_test_pts) * 100

print(f"Test Flagged Points: {flagged_count} / {total_test_pts} ({anomaly_rate:.2f}%)")

# 5. Build Test Scores DataFrame
test_scores_df = pd.DataFrame({
    "time_dt": lgb_df["time_dt"].iloc[split_idx:].values,
    "anomaly_score": scores_te,
    "is_anomaly": is_anomaly
})
for col in sensor_cols:
    test_scores_df[f"{col}_z"] = z_te[col].values

# 6. Group Contiguous Flagged Timestamps into Episodes
flagged_df = test_scores_df[test_scores_df["is_anomaly"]].copy().sort_values("time_dt").reset_index(drop=True)
flagged_df["time_gap"] = flagged_df["time_dt"].diff()
flagged_df["episode_group"] = (flagged_df["time_gap"] != pd.Timedelta(minutes=5)).cumsum()

episodes = []
for ep_id, group in flagged_df.groupby("episode_group"):
    start_t = group["time_dt"].min()
    end_t = group["time_dt"].max()
    dur_mins = int((end_t - start_t).total_seconds() / 60) + 5
    pts_cnt = len(group)
    max_s = group["anomaly_score"].max()
    mean_s = group["anomaly_score"].mean()
    
    # Primary sensor contributor
    z_means = {col: group[f"{col}_z"].abs().mean() for col in sensor_cols}
    primary_sensor = max(z_means, key=z_means.get)
    
    episodes.append({
        "episode_id": ep_id,
        "start_time": start_t,
        "end_time": end_t,
        "duration_minutes": dur_mins,
        "flagged_points": pts_cnt,
        "max_anomaly_score": round(max_s, 4),
        "mean_anomaly_score": round(mean_s, 4),
        "primary_sensor_contributor": primary_sensor
    })

ep_df = pd.DataFrame(episodes)
ep_sorted = ep_df.sort_values("max_anomaly_score", ascending=False).reset_index(drop=True)

print(f"Extracted {len(ep_df)} contiguous abnormal episodes.")
print("\n=== TOP 5 ABNORMAL EPISODES BY MAX ANOMALY SCORE ===")
print(ep_sorted.head(5).to_string(index=False))

# Save CSV Outputs
ep_path = os.path.join(output_dir, "lightgbm_abnormal_episodes.csv")
score_path = os.path.join(output_dir, "lightgbm_test_scores.csv")

ep_sorted.to_csv(ep_path, index=False)
test_scores_df.to_csv(score_path, index=False)

print(f"\nSaved abnormal episodes to: {ep_path}")
print(f"Saved test anomaly scores to: {score_path}")


Leakage-Safe 99th Percentile Threshold (tau): 7.8955
Test Flagged Points: 392 / 74946 (0.52%)
Extracted 307 contiguous abnormal episodes.

=== TOP 5 ABNORMAL EPISODES BY MAX ANOMALY SCORE ===
 episode_id          start_time            end_time  duration_minutes  flagged_points  max_anomaly_score  mean_anomaly_score primary_sensor_contributor
          3 2019-11-20 16:10:00 2019-11-20 16:10:00                 5               1            56.9940             56.9940   Cyclone_Outlet_Gas_draft
         54 2020-01-29 03:05:00 2020-01-29 03:10:00                10               2            50.0363             31.1714   Cyclone_Outlet_Gas_draft
        235 2020-05-21 11:20:00 2020-05-21 11:20:00                 5               1            42.7268             42.7268   Cyclone_Outlet_Gas_draft
        273 2020-07-18 13:00:00 2020-07-18 13:15:00                20               4            40.3379             19.6693        Cyclone_Inlet_Draft
        275 2020-07-19 04:45:00 2020-07-19 04:55

---
## PART 7: LIGHTGBM FEATURE IMPORTANCE ANALYSIS

We extract feature importances (Gain & Split Count) across all 6 sensor models to analyze predictor contributions and inter-sensor dynamic dependencies.

In [13]:
imp_records = []

for target_sensor in sensor_cols:
    model = lgb_models[target_sensor]
    gains = model.booster_.feature_importance(importance_type="gain")
    splits = model.booster_.feature_importance(importance_type="split")
    
    for feat, gain, split in zip(feature_cols, gains, splits):
        imp_records.append({
            "Target_Sensor": target_sensor,
            "Feature": feat,
            "Importance_Gain": gain,
            "Importance_Split": split
        })

imp_df = pd.DataFrame(imp_records)

# Overall Macro Importance (Average Gain across all 6 sensors)
macro_imp = imp_df.groupby("Feature")["Importance_Gain"].mean().reset_index()
macro_imp = macro_imp.sort_values("Importance_Gain", ascending=False).reset_index(drop=True)

print("=== TOP 15 OVERALL MOST IMPORTANT FEATURES (MACRO GAIN) ===")
print(macro_imp.head(15).to_string(index=False))

print("\n=== TOP 3 PREDICTOR FEATURES PER SENSOR ===")
for target_sensor in sensor_cols:
    sub = imp_df[imp_df["Target_Sensor"] == target_sensor].sort_values("Importance_Gain", ascending=False)
    top_3 = sub.head(3)[["Feature", "Importance_Gain"]].values
    top_str = ", ".join([f"{f} ({g:,.0f})" for f, g in top_3])
    print(f"{target_sensor:26s}: {top_str}")

# Save Feature Importance CSV
imp_path = os.path.join(output_dir, "lightgbm_feature_importance.csv")
imp_df.to_csv(imp_path, index=False)
print(f"\nSaved feature importance details to: {imp_path}")

# Plot Top 15 Features Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
top15 = macro_imp.head(15)
ax.barh(top15["Feature"][::-1], top15["Importance_Gain"][::-1] / 1e9, color="#003366")
ax.set_xlabel("Mean Feature Importance (Gain in Billions)", fontsize=11, fontweight="bold")
ax.set_title("Top 15 Overall Most Important Features (LightGBM Multi-Output)", fontsize=13, fontweight="bold", color="#003366")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()


=== TOP 15 OVERALL MOST IMPORTANT FEATURES (MACRO GAIN) ===
                            Feature  Importance_Gain
   Cyclone_Material_Temp_roll3_mean     4.936250e+10
        Cyclone_Inlet_Gas_Temp_lag1     3.011549e+10
       Cyclone_Gas_Outlet_Temp_lag1     2.138496e+10
 Cyclone_Gas_Outlet_Temp_roll3_mean     1.958525e+10
  Cyclone_Inlet_Gas_Temp_roll3_mean     1.514846e+10
       Cyclone_Gas_Outlet_Temp_lag2     8.936625e+09
         Cyclone_Material_Temp_lag1     8.028448e+09
        Cyclone_Inlet_Gas_Temp_lag2     5.674297e+09
      Cyclone_Outlet_Gas_draft_lag1     4.344488e+09
            Cyclone_cone_draft_lag1     2.734078e+09
           Cyclone_Inlet_Draft_lag1     2.689402e+09
      Cyclone_cone_draft_roll3_mean     8.183790e+08
         Cyclone_Material_Temp_lag2     8.017271e+08
Cyclone_Outlet_Gas_draft_roll3_mean     3.207946e+08
        Cyclone_Inlet_Gas_Temp_lag3     1.912428e+08

=== TOP 3 PREDICTOR FEATURES PER SENSOR ===
Cyclone_Inlet_Gas_Temp    : Cyclone_Inlet_Gas_T

C:\Users\Admin\AppData\Local\Temp\ipykernel_1460\2713711888.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# Deployment artifact export - run only after all training and anomaly cells above
import json
from pathlib import Path

import joblib

required_vars = [
    "lgb_models",
    "feature_cols",
    "sensor_cols",
    "mu_tr",
    "std_tr",
    "tau",
]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError(
        "Run the existing training and residual cells first; missing variables: "
        + ", ".join(missing_vars)
    )

if set(lgb_models.keys()) != set(sensor_cols):
    raise RuntimeError("lgb_models keys do not match sensor_cols")
if len(feature_cols) != 53:
    raise RuntimeError(f"Expected 53 ordered features, found {len(feature_cols)}")

artifact_dir = Path("../artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

model_paths = {}
for sensor in sensor_cols:
    model_path = artifact_dir / f"lightgbm_{sensor}.joblib"
    joblib.dump(lgb_models[sensor], model_path)
    model_paths[sensor] = str(model_path)

metadata = {
    "sensor_columns": sensor_cols,
    "feature_columns": feature_cols,
    "feature_count": len(feature_cols),
    "model_paths": model_paths,
    "training_fraction": 0.8,
    "lgbm_params": {
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 6,
        "num_leaves": 31,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    },
    "residual_standardization": {
        "definition": "(training_residual - training_residual_mean) / training_residual_std",
        "ddof": 1,
        "mean_by_sensor": mu_tr.to_dict(),
        "std_by_sensor": std_tr.to_dict(),
    },
    "anomaly_threshold": {
        "value": float(tau),
        "training_percentile": 99,
        "score_definition": "L2 norm of standardized six-sensor residual vector",
    },
}

metadata_path = artifact_dir / "lightgbm_deployment_metadata.json"
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved 6 LightGBM models to: {artifact_dir.resolve()}")
print(f"Saved deployment metadata to: {metadata_path.resolve()}")


Saved 6 LightGBM models to: C:\Users\Admin\Desktop\datascience term1 notes\interview project\katta_Jahnavi_Assignment\artifacts
Saved deployment metadata to: C:\Users\Admin\Desktop\datascience term1 notes\interview project\katta_Jahnavi_Assignment\artifacts\lightgbm_deployment_metadata.json
